# Historical research notebook

Sanitized source archive, not a validated current model. Saved outputs were removed. Data, credentials and model weights are not included. Path placeholders are not automatically expanded. See docs/CODE_CHANGELOG.md.


In [ ]:
# 셀 1: 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 셀 2: CSV 탐색 + 컬럼 구조 출력
import pandas as pd
import os, glob

BASE_DIR = '<DRIVE_ROOT>'

# MLB 관련 파일 전체 찾기
patterns = ['*mlb*', '*MLB*', '*백테스트*', '*statcast*', '*Statcast*']
all_files = []
for p in patterns:
    all_files += glob.glob(os.path.join(BASE_DIR, '**', p), recursive=True)
csvs = sorted(set([f for f in all_files if f.endswith('.csv')]))

print(f'=== CSV {len(csvs)}개 발견 ===\n')

for f in csvs:
    rel = os.path.relpath(f, BASE_DIR)
    size_mb = os.path.getsize(f) / 1024 / 1024
    try:
        df = pd.read_csv(f, nrows=3)
        print(f'--- {rel} ({size_mb:.1f}MB, {len(pd.read_csv(f))}행) ---')
        print(f'컬럼({len(df.columns)}): {list(df.columns)}')
        print(df.head(2).to_string())
    except Exception as e:
        print(f'--- {rel} ({size_mb:.1f}MB) --- 읽기 실패: {e}')
    print()

In [ ]:
# 셀 2: Phase 1 - EDA 전체
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import warnings
warnings.filterwarnings('ignore')

BASE = '<DRIVE_ROOT>/MLB데이터/2025'

# === 데이터 로드 ===
results = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_경기결과.csv')
games = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_경기목록.csv')
pit_game = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_투수개인기록.csv')
bat_game = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_타자개인기록.csv')
weather = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_날씨.csv')
lineup = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_라인업.csv')
bullpen = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_불펜소모도.csv')
umpire = pd.read_csv(f'{BASE}/mlb_2025_심판.csv')
run_total_baseline = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_정규시즌_총득점기준점.csv')
il = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_IL현황.csv')
statcast_p = pd.read_csv(f'{BASE}/mlb_2025_statcast_투수.csv')
statcast_b = pd.read_csv(f'{BASE}/mlb_2025_statcast_타자.csv')
team_rank = pd.read_csv(f'{BASE}/mlb_2025_팀순위.csv')
season_pit = pd.read_csv(f'{BASE}/mlb_2025_선수시즌기록_투수.csv')
season_bat = pd.read_csv(f'{BASE}/mlb_2025_선수시즌기록_타자.csv')

print("=== 전체 데이터 로드 완료 ===")
for name, d in [('경기결과',results),('경기목록',games),('투수경기별',pit_game),
                ('타자경기별',bat_game),('날씨',weather),('라인업',lineup),
                ('불펜소모도',bullpen),('심판',umpire),('총득점기준점',run_total_baseline),
                ('IL현황',il),('statcast투수',statcast_p),('statcast타자',statcast_b),
                ('팀순위',team_rank),('시즌투수',season_pit),('시즌타자',season_bat)]:
    print(f'  {name}: {d.shape}')

In [ ]:
# 셀 3: 기본 테이블 구성 + 타겟 분포
results['날짜'] = pd.to_datetime(results['날짜'])
results['월'] = results['날짜'].dt.month
results['홈승'] = (results['홈팀결과'] == '승').astype(int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1) 홈승률 전체
ax = axes[0,0]
home_wr = results['홈승'].mean()
ax.bar(['Away Win','Home Win'], [1-home_wr, home_wr], color=['#e74c3c','#2ecc71'])
ax.set_title(f'Overall Home Win Rate: {home_wr:.3f}')
ax.set_ylim(0, 0.7)
for i, v in enumerate([1-home_wr, home_wr]):
    ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')

# 2) 월별 홈승률
ax = axes[0,1]
monthly = results.groupby('월')['홈승'].agg(['mean','count'])
ax.bar(monthly.index, monthly['mean'], color='#3498db', alpha=0.8)
ax.axhline(home_wr, color='red', linestyle='--', label=f'Avg {home_wr:.3f}')
ax.set_title('Home Win Rate by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Win Rate')
ax.legend()
for i, row in monthly.iterrows():
    ax.text(i, row['mean']+0.005, f'{row["mean"]:.3f}\n(n={int(row["count"])})',
            ha='center', fontsize=8)

# 3) 총득점 분포
ax = axes[0,2]
ax.hist(results['총득점'], bins=range(0,25), color='#9b59b6', alpha=0.8, edgecolor='white')
ax.axvline(results['총득점'].mean(), color='red', linestyle='--',
           label=f'Mean: {results["총득점"].mean():.1f}')
ax.set_title('Total Runs Distribution')
ax.set_xlabel('Total Runs')
ax.legend()

# 4) 점수차 분포
ax = axes[1,0]
ax.hist(results['점수차'], bins=range(0,16), color='#e67e22', alpha=0.8, edgecolor='white')
ax.axvline(results['점수차'].mean(), color='red', linestyle='--',
           label=f'Mean: {results["점수차"].mean():.1f}')
ax.set_title('Run Differential Distribution')
ax.set_xlabel('Run Diff')
ax.legend()

# 5) 팀별 홈승률 Top/Bottom
ax = axes[1,1]
team_home = results.groupby('홈팀')['홈승'].mean().sort_values()
colors = ['#e74c3c' if v < 0.45 else '#2ecc71' if v > 0.55 else '#95a5a6' for v in team_home.values]
ax.barh(range(len(team_home)), team_home.values, color=colors)
ax.set_yticks(range(len(team_home)))
ax.set_yticklabels([t[:12] for t in team_home.index], fontsize=6)
ax.axvline(0.5, color='black', linestyle='--')
ax.set_title('Home Win Rate by Team')
ax.set_xlim(0.3, 0.7)

# 6) 월별 경기수 + 평균 총득점
ax = axes[1,2]
monthly_runs = results.groupby('월')['총득점'].mean()
ax2 = ax.twinx()
ax.bar(monthly.index, monthly['count'], color='#bdc3c7', alpha=0.6, label='Games')
ax2.plot(monthly_runs.index, monthly_runs.values, 'ro-', label='Avg Runs')
ax.set_title('Games Count & Avg Total Runs by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Games')
ax2.set_ylabel('Avg Total Runs')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('/content/eda_phase1.png', dpi=150, bbox_inches='tight')
plt.show()

# 수치 요약
print("\n=== 수치 요약 ===")
print(f"전체 경기수: {len(results)}")
print(f"홈승률: {home_wr:.4f}")
print(f"평균 총득점: {results['총득점'].mean():.2f} (std: {results['총득점'].std():.2f})")
print(f"평균 점수차: {results['점수차'].mean():.2f}")
print(f"\n월별 상세:")
print(monthly.to_string())

In [ ]:
# 셀 4: 결측치 전체 스캔 + 데이터 조인 가능성 체크
print("=== 핵심 CSV 결측치 현황 ===\n")

for name, d in [('경기결과',results),('경기목록',games),('투수경기별',pit_game),
                ('타자경기별',bat_game),('날씨',weather),('라인업',lineup),
                ('불펜소모도',bullpen),('총득점기준점',run_total_baseline),('statcast투수',statcast_p)]:
    missing = d.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f'--- {name} ---')
        for col, cnt in missing.items():
            print(f'  {col}: {cnt} ({cnt/len(d)*100:.1f}%)')
        print()

# 경기목록-경기결과 조인 체크
merged = games.merge(results, on='gamePk', how='left')
print(f"\n=== 조인 체크 ===")
print(f"경기목록: {len(games)}행")
print(f"경기결과: {len(results)}행")
print(f"inner join: {len(games.merge(results, on='gamePk', how='inner'))}행")
print(f"경기결과에만 있는 gamePk: {len(set(results.gamePk) - set(games.gamePk))}")
print(f"경기목록에만 있는 gamePk: {len(set(games.gamePk) - set(results.gamePk))}")

# 선발투수ID 매칭률
pit_ids = set(pit_game['선수ID'].unique())
home_sp_match = games['홈팀선발ID'].isin(pit_ids).mean()
away_sp_match = games['원정팀선발ID'].isin(pit_ids).mean()
print(f"\n선발투수ID → 투수경기별 매칭률:")
print(f"  홈팀선발: {home_sp_match:.3f}")
print(f"  원정팀선발: {away_sp_match:.3f}")

# Statcast 매칭률
sc_pids = set(statcast_p['player_id'].unique())
sp_ids_in_games = set(games['홈팀선발ID'].unique()) | set(games['원정팀선발ID'].unique())
sc_match = len(sp_ids_in_games & sc_pids) / len(sp_ids_in_games)
print(f"\n선발투수 → Statcast 매칭률: {sc_match:.3f} ({len(sp_ids_in_games & sc_pids)}/{len(sp_ids_in_games)})")

In [ ]:
# 셀 5 수정: 상단 기본 테이블 부분만 교체
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

BASE = '<DRIVE_ROOT>/MLB데이터/2025'

# === 기본 테이블 ===
results = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_경기결과.csv')
games = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_경기목록.csv')
results['날짜'] = pd.to_datetime(results['날짜'])
results['홈승'] = (results['홈팀결과'] == '승').astype(int)

# games에도 날짜 있으므로 results에서는 날짜 제외하고 merge
df = games.merge(results[['gamePk','홈팀득점','원정팀득점','홈팀결과','총득점','홈승']],
                 on='gamePk', how='inner')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.sort_values('날짜').reset_index(drop=True)
print(f'기본 테이블: {df.shape}')
print(f'컬럼: {list(df.columns)}')

# ============================================================
# 피처 1~6: 선발투수 rolling ERA/WHIP/K9
# ============================================================
pit = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_투수개인기록.csv')

# 선발 = gamePk+팀 기준 이닝 최대인 투수
starter = pit.sort_values('이닝', ascending=False).groupby(['gamePk','팀']).first().reset_index()
starter = starter.merge(results[['gamePk','날짜']], on='gamePk')
starter['날짜'] = pd.to_datetime(starter['날짜'])
starter = starter.sort_values(['선수ID','날짜']).reset_index(drop=True)

# expanding rolling (shift로 look-ahead 방지)
for pid, grp in starter.groupby('선수ID'):
    idx = grp.index
    cum_er = grp['자책'].expanding().sum().shift(1)
    cum_ip = grp['이닝'].expanding().sum().shift(1)
    cum_h = grp['피안타'].expanding().sum().shift(1)
    cum_bb = grp['볼넷'].expanding().sum().shift(1)
    cum_k = grp['삼진'].expanding().sum().shift(1)
    starter.loc[idx, 'roll_ERA'] = (cum_er / cum_ip * 9).replace([np.inf, -np.inf], np.nan)
    starter.loc[idx, 'roll_WHIP'] = ((cum_h + cum_bb) / cum_ip).replace([np.inf, -np.inf], np.nan)
    starter.loc[idx, 'roll_K9'] = (cum_k / cum_ip * 9).replace([np.inf, -np.inf], np.nan)

sp_feat = starter[['gamePk','선수ID','roll_ERA','roll_WHIP','roll_K9']].copy()

# 홈선발
df = df.merge(sp_feat.rename(columns={'roll_ERA':'home_sp_ERA','roll_WHIP':'home_sp_WHIP','roll_K9':'home_sp_K9'}),
              left_on=['gamePk','홈팀선발ID'], right_on=['gamePk','선수ID'], how='left').drop(columns='선수ID')
# 원정선발
df = df.merge(sp_feat.rename(columns={'roll_ERA':'away_sp_ERA','roll_WHIP':'away_sp_WHIP','roll_K9':'away_sp_K9'}),
              left_on=['gamePk','원정팀선발ID'], right_on=['gamePk','선수ID'], how='left').drop(columns='선수ID')

# ============================================================
# 피처 7: ERA 스프레드
# ============================================================
df['era_spread'] = df['away_sp_ERA'] - df['home_sp_ERA']

# ============================================================
# 피처 8-9: 팀 최근 10경기 승률
# ============================================================
team_games = []
for _, row in results.iterrows():
    team_games.append({'gamePk': row['gamePk'], '날짜': row['날짜'], '팀': row['홈팀'], '승': int(row['홈팀결과']=='승')})
    team_games.append({'gamePk': row['gamePk'], '날짜': row['날짜'], '팀': row['원정팀'], '승': int(row['원정팀결과']=='승')})
tg = pd.DataFrame(team_games).sort_values(['팀','날짜'])

tg['roll_wr10'] = tg.groupby('팀')['승'].transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
team_wr = tg[['gamePk','팀','roll_wr10']].copy()

df = df.merge(team_wr.rename(columns={'roll_wr10':'home_wr10'}),
              left_on=['gamePk','홈팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
df = df.merge(team_wr.rename(columns={'roll_wr10':'away_wr10'}),
              left_on=['gamePk','원정팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
df['wr_spread'] = df['home_wr10'] - df['away_wr10']

# ============================================================
# 피처 10-11: 라인업 평균 OPS (rolling)
# ============================================================
lineup = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_라인업.csv')
bat = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_타자개인기록.csv')

# 타자별 경기 OPS (당일 기준 expanding, shift)
bat = bat.merge(results[['gamePk','날짜']], on='gamePk')
bat['날짜'] = pd.to_datetime(bat['날짜'])
bat = bat.sort_values(['선수ID','날짜']).reset_index(drop=True)
# AB 기반 누적 OPS
bat['AB'] = bat['타수']
bat['H'] = bat['안타']
bat['BB'] = bat['볼넷']
bat['PA'] = bat['타석']
bat['TB'] = bat['루타']

for pid, grp in bat.groupby('선수ID'):
    idx = grp.index
    ch = grp['H'].expanding().sum().shift(1)
    cab = grp['AB'].expanding().sum().shift(1)
    cbb = grp['BB'].expanding().sum().shift(1)
    cpa = grp['PA'].expanding().sum().shift(1)
    ctb = grp['TB'].expanding().sum().shift(1)
    avg = ch / cab
    obp = (ch + cbb) / cpa
    slg = ctb / cab
    bat.loc[idx, 'roll_OPS'] = (obp + slg).replace([np.inf, -np.inf], np.nan)

bat_ops = bat[['gamePk','선수ID','roll_OPS']].copy()

# 라인업과 조인
lineup['날짜'] = pd.to_datetime(lineup['날짜'])
lineup_ops = lineup.merge(bat_ops, left_on=['gamePk','선수ID'], right_on=['gamePk','선수ID'], how='left')
lineup_avg_ops = lineup_ops.groupby(['gamePk','홈원정'])['roll_OPS'].mean().reset_index()

home_ops = lineup_avg_ops[lineup_avg_ops['홈원정']=='home'][['gamePk','roll_OPS']].rename(columns={'roll_OPS':'home_lineup_OPS'})
away_ops = lineup_avg_ops[lineup_avg_ops['홈원정']=='away'][['gamePk','roll_OPS']].rename(columns={'roll_OPS':'away_lineup_OPS'})
df = df.merge(home_ops, on='gamePk', how='left')
df = df.merge(away_ops, on='gamePk', how='left')
df['ops_spread'] = df['home_lineup_OPS'] - df['away_lineup_OPS']

# ============================================================
# 피처 12-13: 불펜 위험도
# ============================================================
bp = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_불펜소모도.csv')
bp['위험'] = bp['위험등급'].str.contains('고위험|주의', na=False).astype(int)
bp_risk = bp.groupby(['gamePk','팀명'])['위험'].mean().reset_index()

df = df.merge(bp_risk.rename(columns={'팀명':'_t','위험':'home_bp_risk'}),
              left_on=['gamePk','홈팀'], right_on=['gamePk','_t'], how='left').drop(columns='_t')
df = df.merge(bp_risk.rename(columns={'팀명':'_t','위험':'away_bp_risk'}),
              left_on=['gamePk','원정팀'], right_on=['gamePk','_t'], how='left').drop(columns='_t')

# ============================================================
# 피처 14-15: 날씨
# ============================================================
weather = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_날씨.csv')
weather['돔구장_int'] = weather['돔구장'].astype(int)
weather['바람효과_수치'] = weather['바람효과'].map({'타자유리':1,'중립':0,'투수유리':-1}).fillna(0)
df = df.merge(weather[['gamePk','돔구장_int','기온_F','바람속도_mph','바람효과_수치']], on='gamePk', how='left')

# ============================================================
# 피처 16: 구장 Park Factor (시즌 평균득점 기준)
# ============================================================
park_rpg = results.groupby('구장')['총득점'].mean().reset_index()
park_rpg.columns = ['구장','park_rpg']
df = df.merge(park_rpg, on='구장', how='left')

# ============================================================
# 피처 17: Statcast xERA (시즌값, 결측 많을 수 있음)
# ============================================================
sc_p = pd.read_csv(f'{BASE}/mlb_2025_statcast_투수.csv')
sc_xera = sc_p[['player_id','xera']].copy()
df = df.merge(sc_xera.rename(columns={'player_id':'홈팀선발ID','xera':'home_xera'}),
              on='홈팀선발ID', how='left')
df = df.merge(sc_xera.rename(columns={'player_id':'원정팀선발ID','xera':'away_xera'}),
              on='원정팀선발ID', how='left')
df['xera_spread'] = df['away_xera'] - df['home_xera']

# ============================================================
# 피처 18: 심판 RPG
# ============================================================
ump = pd.read_csv(f'{BASE}/mlb_2025_심판.csv')
hp_ump = ump[ump['포지션']=='Home Plate'][['gamePk','심판명','심판ID']].copy()
hp_ump = hp_ump.merge(results[['gamePk','날짜','총득점']], on='gamePk')
hp_ump['날짜'] = pd.to_datetime(hp_ump['날짜'])
hp_ump = hp_ump.sort_values(['심판ID','날짜'])

for uid, grp in hp_ump.groupby('심판ID'):
    idx = grp.index
    hp_ump.loc[idx, 'ump_rpg'] = grp['총득점'].expanding().mean().shift(1)

df = df.merge(hp_ump[['gamePk','ump_rpg']], on='gamePk', how='left')

print(f'\n최종 피처 테이블: {df.shape}')
print(f'피처 목록 생성 완료')

In [ ]:
# 셀 6: 단변량 AUC + 시각화
feature_cols = [
    'home_sp_ERA', 'away_sp_ERA', 'home_sp_WHIP', 'away_sp_WHIP',
    'home_sp_K9', 'away_sp_K9', 'era_spread',
    'home_wr10', 'away_wr10', 'wr_spread',
    'home_lineup_OPS', 'away_lineup_OPS', 'ops_spread',
    'home_bp_risk', 'away_bp_risk',
    '돔구장_int', '기온_F', '바람속도_mph', '바람효과_수치',
    'park_rpg',
    'home_xera', 'away_xera', 'xera_spread',
    'ump_rpg'
]

# --- 단변량 AUC 계산 ---
auc_results = []
for col in feature_cols:
    valid = df[[col, '홈승']].dropna()
    if len(valid) < 100:
        auc_results.append({'feature': col, 'auc': np.nan, 'n': len(valid), 'direction': '?'})
        continue
    try:
        auc = roc_auc_score(valid['홈승'], valid[col])
        direction = 'positive' if auc >= 0.5 else 'negative'
        auc_adj = max(auc, 1-auc)  # 방향 보정
        auc_results.append({'feature': col, 'auc': auc, 'auc_adj': auc_adj,
                           'n': len(valid), 'direction': direction})
    except:
        auc_results.append({'feature': col, 'auc': np.nan, 'n': len(valid), 'direction': '?'})

auc_df = pd.DataFrame(auc_results).sort_values('auc_adj', ascending=False)
print("=== 단변량 AUC (방향 보정) ===")
print(auc_df[['feature','auc','auc_adj','n','direction']].to_string(index=False))

# --- 시각화: 상위 12개 피처 boxplot ---
top12 = auc_df.head(12)['feature'].tolist()
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(top12):
    ax = axes[i]
    valid = df[[col, '홈승']].dropna()
    win_vals = valid[valid['홈승']==1][col]
    lose_vals = valid[valid['홈승']==0][col]

    ax.boxplot([lose_vals, win_vals], labels=['Away Win', 'Home Win'],
               patch_artist=True,
               boxprops=[dict(facecolor='#e74c3c', alpha=0.7),
                         dict(facecolor='#2ecc71', alpha=0.7)][0:1]*2)
    # 직접 색 지정
    bp_plot = ax.boxplot([lose_vals, win_vals], labels=['Away Win', 'Home Win'], patch_artist=True)
    bp_plot['boxes'][0].set_facecolor('#e74c3c')
    bp_plot['boxes'][0].set_alpha(0.7)
    bp_plot['boxes'][1].set_facecolor('#2ecc71')
    bp_plot['boxes'][1].set_alpha(0.7)

    auc_val = auc_df[auc_df['feature']==col]['auc_adj'].values[0]
    ax.set_title(f'{col}\nAUC={auc_val:.4f}', fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle('Top 12 Features: Home Win vs Away Win Distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('/content/eda_phase2_univariate.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 스프레드 피처 KDE ---
spread_cols = ['era_spread', 'wr_spread', 'ops_spread', 'xera_spread']
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, col in enumerate(spread_cols):
    ax = axes[i]
    valid = df[[col, '홈승']].dropna()
    valid[valid['홈승']==1][col].plot.kde(ax=ax, label='Home Win', color='#2ecc71', linewidth=2)
    valid[valid['홈승']==0][col].plot.kde(ax=ax, label='Away Win', color='#e74c3c', linewidth=2)
    ax.set_title(f'{col} KDE')
    ax.legend(fontsize=8)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/eda_phase2_kde.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 결측률 요약 ---
print("\n=== 피처 결측률 ===")
for col in feature_cols:
    miss = df[col].isnull().mean()
    print(f'  {col}: {miss:.3f} ({int(df[col].isnull().sum())}건)')

In [ ]:
# 셀 6 수정: 단변량 시각화
feature_cols = [
    'home_sp_ERA', 'away_sp_ERA', 'home_sp_WHIP', 'away_sp_WHIP',
    'home_sp_K9', 'away_sp_K9', 'era_spread',
    'home_wr10', 'away_wr10', 'wr_spread',
    'home_lineup_OPS', 'away_lineup_OPS', 'ops_spread',
    'home_bp_risk', 'away_bp_risk',
    '돔구장_int', '기온_F', '바람속도_mph', '바람효과_수치',
    'park_rpg',
    'home_xera', 'away_xera', 'xera_spread',
    'ump_rpg'
]

# 중복 제거 (merge에서 1:N으로 부풀어난 경우)
print(f'중복 제거 전: {df.shape}')
df = df.drop_duplicates(subset='gamePk', keep='first').reset_index(drop=True)
print(f'중복 제거 후: {df.shape}')

# --- 단변량 AUC 계산 ---
auc_results = []
for col in feature_cols:
    valid = df[[col, '홈승']].dropna()
    if len(valid) < 100:
        auc_results.append({'feature': col, 'auc': np.nan, 'auc_adj': np.nan, 'n': len(valid), 'direction': '?'})
        continue
    try:
        auc = roc_auc_score(valid['홈승'], valid[col])
        direction = 'positive' if auc >= 0.5 else 'negative'
        auc_adj = max(auc, 1-auc)
        auc_results.append({'feature': col, 'auc': auc, 'auc_adj': auc_adj,
                           'n': len(valid), 'direction': direction})
    except:
        auc_results.append({'feature': col, 'auc': np.nan, 'auc_adj': np.nan, 'n': len(valid), 'direction': '?'})

auc_df = pd.DataFrame(auc_results).sort_values('auc_adj', ascending=False)
print("\n=== 단변량 AUC (방향 보정) ===")
print(auc_df[['feature','auc','auc_adj','n','direction']].to_string(index=False))

# --- 시각화: 상위 12개 피처 boxplot ---
top12 = auc_df.head(12)['feature'].tolist()
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(top12):
    ax = axes[i]
    valid = df[[col, '홈승']].dropna()
    lose_vals = valid[valid['홈승']==0][col]
    win_vals = valid[valid['홈승']==1][col]

    bp_plot = ax.boxplot([lose_vals, win_vals], labels=['Away Win', 'Home Win'], patch_artist=True)
    bp_plot['boxes'][0].set_facecolor('#e74c3c')
    bp_plot['boxes'][0].set_alpha(0.7)
    bp_plot['boxes'][1].set_facecolor('#2ecc71')
    bp_plot['boxes'][1].set_alpha(0.7)

    auc_val = auc_df[auc_df['feature']==col]['auc_adj'].values[0]
    ax.set_title(f'{col}\nAUC={auc_val:.4f}', fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle('Top 12 Features: Home Win vs Away Win Distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('/content/eda_phase2_univariate.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 스프레드 피처 KDE ---
spread_cols = ['era_spread', 'wr_spread', 'ops_spread', 'xera_spread']
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, col in enumerate(spread_cols):
    ax = axes[i]
    valid = df[[col, '홈승']].dropna()
    valid[valid['홈승']==1][col].plot.kde(ax=ax, label='Home Win', color='#2ecc71', linewidth=2)
    valid[valid['홈승']==0][col].plot.kde(ax=ax, label='Away Win', color='#e74c3c', linewidth=2)
    ax.set_title(f'{col} KDE')
    ax.legend(fontsize=8)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/eda_phase2_kde.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 결측률 요약 ---
print("\n=== 피처 결측률 ===")
for col in feature_cols:
    miss = df[col].isnull().mean()
    print(f'  {col}: {miss:.3f} ({int(df[col].isnull().sum())}건)')

In [ ]:
# 셀 7: Phase 3 - 상관행렬 + VIF + 피처 선정
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

# 유효 피처 (AUC 0.52 이상만)
candidates = [
    'xera_spread', 'home_xera', 'away_xera',
    'wr_spread', 'home_wr10', 'away_wr10',
    'ops_spread', 'home_lineup_OPS', 'away_lineup_OPS',
    'home_sp_K9', 'away_sp_K9',
    'era_spread', 'home_sp_ERA', 'away_sp_ERA',
    'home_sp_WHIP', 'away_sp_WHIP',
    'park_rpg', '기온_F',
    'home_bp_risk'
]

# 1) 상관행렬 히트맵
valid = df[candidates + ['홈승']].dropna()
print(f'피처 후보 {len(candidates)}개, 결측 없는 행: {len(valid)} / {len(df)}')

corr = valid[candidates].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size':7})
ax.set_title('Feature Correlation Matrix (candidates)', fontsize=14)
plt.tight_layout()
plt.savefig('/content/eda_phase3_corr.png', dpi=150, bbox_inches='tight')
plt.show()

# 2) 타겟과의 상관
target_corr = valid[candidates].corrwith(valid['홈승']).abs().sort_values(ascending=False)
print("\n=== 타겟(홈승)과의 절대 상관 ===")
print(target_corr.to_string())

# 3) VIF 계산
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# 스프레드 vs 개별 피처 중복 제거 → 스프레드 우선
vif_candidates = [
    'xera_spread',      # home_xera + away_xera 대체
    'wr_spread',        # home_wr10 + away_wr10 대체
    'ops_spread',       # home/away lineup OPS 대체
    'era_spread',       # home/away ERA 대체
    'home_sp_K9', 'away_sp_K9',
    'home_sp_WHIP', 'away_sp_WHIP',
    'park_rpg', '기온_F',
    'home_bp_risk'
]

vif_valid = df[vif_candidates].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(vif_valid)

print("\n=== VIF (스프레드 중심) ===")
vif_data = []
for i, col in enumerate(vif_candidates):
    vif = variance_inflation_factor(X_scaled, i)
    vif_data.append({'feature': col, 'VIF': round(vif, 2)})
    print(f'  {col}: {vif:.2f}')
vif_df = pd.DataFrame(vif_data)

# 4) 스프레드 vs 개별 피처 비교
print("\n=== 스프레드 vs 개별 피처 AUC 비교 ===")
comparisons = [
    ('xera_spread', ['home_xera', 'away_xera']),
    ('era_spread', ['home_sp_ERA', 'away_sp_ERA']),
    ('wr_spread', ['home_wr10', 'away_wr10']),
    ('ops_spread', ['home_lineup_OPS', 'away_lineup_OPS']),
]
for run_margin, individuals in comparisons:
    s_valid = df[[run_margin, '홈승']].dropna()
    s_auc = roc_auc_score(s_valid['홈승'], s_valid[run_margin])
    s_auc = max(s_auc, 1-s_auc)
    print(f'\n  {run_margin}: AUC={s_auc:.4f} (n={len(s_valid)})')
    for ind in individuals:
        i_valid = df[[ind, '홈승']].dropna()
        i_auc = roc_auc_score(i_valid['홈승'], i_valid[ind])
        i_auc = max(i_auc, 1-i_auc)
        print(f'    {ind}: AUC={i_auc:.4f} (n={len(i_valid)})')

# 5) xera_spread + era_spread fallback 테스트
df['era_combined'] = df['xera_spread'].fillna(df['era_spread'])
combined_valid = df[['era_combined','홈승']].dropna()
combined_auc = roc_auc_score(combined_valid['홈승'], combined_valid['era_combined'])
combined_auc = max(combined_auc, 1-combined_auc)
print(f'\n=== Fallback 피처: xera→era combined ===')
print(f'  AUC={combined_auc:.4f} (n={len(combined_valid)})')
print(f'  결측률: {df["era_combined"].isnull().mean():.3f}')

In [ ]:
# 셀 8: Phase 3 보충 - 피처 간 산점도 (핵심 3개)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# xera_spread vs era_spread (중복도 확인)
ax = axes[0]
valid = df[['xera_spread','era_spread','홈승']].dropna()
ax.scatter(valid[valid['홈승']==0]['xera_spread'], valid[valid['홈승']==0]['era_spread'],
           alpha=0.3, c='#e74c3c', s=10, label='Away Win')
ax.scatter(valid[valid['홈승']==1]['xera_spread'], valid[valid['홈승']==1]['era_spread'],
           alpha=0.3, c='#2ecc71', s=10, label='Home Win')
r = valid['xera_spread'].corr(valid['era_spread'])
ax.set_title(f'xera_spread vs era_spread (r={r:.3f})')
ax.set_xlabel('xera_spread')
ax.set_ylabel('era_spread')
ax.legend()

# xera_spread vs wr_spread (독립성 확인)
ax = axes[1]
valid = df[['xera_spread','wr_spread','홈승']].dropna()
ax.scatter(valid[valid['홈승']==0]['xera_spread'], valid[valid['홈승']==0]['wr_spread'],
           alpha=0.3, c='#e74c3c', s=10, label='Away Win')
ax.scatter(valid[valid['홈승']==1]['xera_spread'], valid[valid['홈승']==1]['wr_spread'],
           alpha=0.3, c='#2ecc71', s=10, label='Home Win')
r = valid['xera_spread'].corr(valid['wr_spread'])
ax.set_title(f'xera_spread vs wr_spread (r={r:.3f})')
ax.set_xlabel('xera_spread')
ax.set_ylabel('wr_spread')
ax.legend()

# ops_spread vs wr_spread
ax = axes[2]
valid = df[['ops_spread','wr_spread','홈승']].dropna()
ax.scatter(valid[valid['홈승']==0]['ops_spread'], valid[valid['홈승']==0]['wr_spread'],
           alpha=0.3, c='#e74c3c', s=10, label='Away Win')
ax.scatter(valid[valid['홈승']==1]['ops_spread'], valid[valid['홈승']==1]['wr_spread'],
           alpha=0.3, c='#2ecc71', s=10, label='Home Win')
r = valid['ops_spread'].corr(valid['wr_spread'])
ax.set_title(f'ops_spread vs wr_spread (r={r:.3f})')
ax.set_xlabel('ops_spread')
ax.set_ylabel('wr_spread')
ax.legend()

plt.tight_layout()
plt.savefig('/content/eda_phase3_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 셀 9: 추가 피처 생성 (피치데이터, 홈원정ERA, IL, 트렌드, 스트릭)
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

BASE = '<DRIVE_ROOT>/MLB데이터/2025'
results = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_경기결과.csv')
results['날짜'] = pd.to_datetime(results['날짜'])
results['홈승'] = (results['홈팀결과'] == '승').astype(int)

# df는 이전 셀에서 이어받음 (2423행, 40컬럼)
print(f'시작 테이블: {df.shape}')

# ============================================================
# 추가1: 홈/원정 분리 ERA (A안 핵심 피처)
# ============================================================
pit = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_투수개인기록.csv')
pit = pit.merge(results[['gamePk','날짜']], on='gamePk')
pit['날짜'] = pd.to_datetime(pit['날짜'])

# 선발 = gamePk+팀 기준 이닝 최대
starter = pit.sort_values('이닝', ascending=False).groupby(['gamePk','팀']).first().reset_index()
starter = starter.sort_values(['선수ID','날짜']).reset_index(drop=True)

# 홈/원정 태그
games_ha = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_경기목록.csv')
home_map = games_ha.set_index('gamePk')['홈팀'].to_dict()
starter['is_home'] = starter.apply(lambda r: 1 if home_map.get(r['gamePk'])==r['팀'] else 0, axis=1)

# 홈경기만/원정경기만 누적 ERA
for pid, grp in starter.groupby('선수ID'):
    idx = grp.index
    # 홈경기만
    home_mask = grp['is_home'] == 1
    home_grp = grp[home_mask]
    if len(home_grp) > 0:
        h_er = home_grp['자책'].expanding().sum().shift(1)
        h_ip = home_grp['이닝'].expanding().sum().shift(1)
        starter.loc[home_grp.index, 'roll_home_ERA'] = (h_er / h_ip * 9).replace([np.inf,-np.inf], np.nan)
    # 원정경기만
    away_mask = grp['is_home'] == 0
    away_grp = grp[away_mask]
    if len(away_grp) > 0:
        a_er = away_grp['자책'].expanding().sum().shift(1)
        a_ip = away_grp['이닝'].expanding().sum().shift(1)
        starter.loc[away_grp.index, 'roll_away_ERA'] = (a_er / a_ip * 9).replace([np.inf,-np.inf], np.nan)

# 홈팀 선발의 홈ERA, 원정팀 선발의 원정ERA
sp_home_era = starter[starter['is_home']==1][['gamePk','선수ID','roll_home_ERA']].copy()
sp_away_era = starter[starter['is_home']==0][['gamePk','선수ID','roll_away_ERA']].copy()

df = df.merge(sp_home_era.rename(columns={'roll_home_ERA':'home_sp_homeERA'}),
              left_on=['gamePk','홈팀선발ID'], right_on=['gamePk','선수ID'], how='left').drop(columns='선수ID')
df = df.merge(sp_away_era.rename(columns={'roll_away_ERA':'away_sp_awayERA'}),
              left_on=['gamePk','원정팀선발ID'], right_on=['gamePk','선수ID'], how='left').drop(columns='선수ID')
df['ha_era_spread'] = df['away_sp_awayERA'] - df['home_sp_homeERA']

# ============================================================
# 추가2: 팀 최근5경기 평균 득점/실점
# ============================================================
team_games = []
for _, row in results.iterrows():
    team_games.append({'gamePk':row['gamePk'],'날짜':row['날짜'],'팀':row['홈팀'],
                       '득점':row['홈팀득점'],'실점':row['원정팀득점']})
    team_games.append({'gamePk':row['gamePk'],'날짜':row['날짜'],'팀':row['원정팀'],
                       '득점':row['원정팀득점'],'실점':row['홈팀득점']})
tg = pd.DataFrame(team_games).sort_values(['팀','날짜'])
tg['recent5_득점'] = tg.groupby('팀')['득점'].transform(lambda x: x.shift(1).rolling(5, min_periods=3).mean())
tg['recent5_실점'] = tg.groupby('팀')['실점'].transform(lambda x: x.shift(1).rolling(5, min_periods=3).mean())
tg['run_diff5'] = tg['recent5_득점'] - tg['recent5_실점']

tr = tg[['gamePk','팀','recent5_득점','recent5_실점','run_diff5']].copy()
df = df.merge(tr.rename(columns={'recent5_득점':'home_r5_득점','recent5_실점':'home_r5_실점','run_diff5':'home_rdiff5'}),
              left_on=['gamePk','홈팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
df = df.merge(tr.rename(columns={'recent5_득점':'away_r5_득점','recent5_실점':'away_r5_실점','run_diff5':'away_rdiff5'}),
              left_on=['gamePk','원정팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
df['rdiff_spread'] = df['home_rdiff5'] - df['away_rdiff5']

# ============================================================
# 추가3: 연승/연패 스트릭
# ============================================================
tg2 = pd.DataFrame(team_games).sort_values(['팀','날짜'])
tg2['승'] = (tg2['득점'] > tg2['실점']).astype(int)

def calc_streak(series):
    streak = []
    current = 0
    for val in series:
        if val == 1:
            current = max(current, 0) + 1
        else:
            current = min(current, 0) - 1
        streak.append(current)
    return streak

tg2['streak'] = tg2.groupby('팀')['승'].transform(lambda x: calc_streak(x.values))
tg2['streak_prev'] = tg2.groupby('팀')['streak'].shift(1)
streak_feat = tg2[['gamePk','팀','streak_prev']].copy()

df = df.merge(streak_feat.rename(columns={'streak_prev':'home_streak'}),
              left_on=['gamePk','홈팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
df = df.merge(streak_feat.rename(columns={'streak_prev':'away_streak'}),
              left_on=['gamePk','원정팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
df['streak_spread'] = df['home_streak'] - df['away_streak']

# ============================================================
# 추가4: IL(부상) 주전급 부상자 수
# ============================================================
il = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_IL현황.csv')
il_count = il.groupby(['gamePk','팀명']).size().reset_index(name='il_count')

df = df.merge(il_count.rename(columns={'팀명':'_t','il_count':'home_il'}),
              left_on=['gamePk','홈팀'], right_on=['gamePk','_t'], how='left').drop(columns='_t')
df = df.merge(il_count.rename(columns={'팀명':'_t','il_count':'away_il'}),
              left_on=['gamePk','원정팀'], right_on=['gamePk','_t'], how='left').drop(columns='_t')
df['il_spread'] = df['away_il'] - df['home_il']

# ============================================================
# 추가5: Statcast 타자 barrel rate (라인업 평균)
# ============================================================
sc_b = pd.read_csv(f'{BASE}/mlb_2025_statcast_타자.csv')
lineup = pd.read_csv(f'{BASE}/백테스트_라인업/mlb_2025_백테스트_라인업.csv')

lineup_brl = lineup.merge(sc_b[['player_id','brl_percent']].rename(columns={'player_id':'선수ID'}),
                          on='선수ID', how='left')
lineup_brl_avg = lineup_brl.groupby(['gamePk','홈원정'])['brl_percent'].mean().reset_index()

home_brl = lineup_brl_avg[lineup_brl_avg['홈원정']=='home'][['gamePk','brl_percent']].rename(columns={'brl_percent':'home_brl'})
away_brl = lineup_brl_avg[lineup_brl_avg['홈원정']=='away'][['gamePk','brl_percent']].rename(columns={'brl_percent':'away_brl'})
df = df.merge(home_brl, on='gamePk', how='left')
df = df.merge(away_brl, on='gamePk', how='left')
df['brl_spread'] = df['home_brl'] - df['away_brl']

# ============================================================
# 추가6: 피치데이터 - 선발 최근3경기 평균구속, 헛스윙률
# ============================================================
print("피치데이터 로딩 중... (71만행)")
pitch = pd.read_csv(f'{BASE}/mlb_2025_정규시즌_피치데이터.csv')
pitch = pitch.merge(results[['gamePk','날짜']], on='gamePk')
pitch['날짜'] = pd.to_datetime(pitch['날짜'])
print(f"피치데이터: {pitch.shape}")

# 선발투수 = 1이닝에 등판한 투수
first_inning = pitch[pitch['이닝'] <= 1].groupby(['gamePk','투수ID']).size().reset_index(name='cnt')
first_inning = first_inning.sort_values('cnt', ascending=False).groupby('gamePk').first().reset_index()

# 경기별 선발투수의 평균구속, 헛스윙률
swstr_map = {'Swinging Strike': 1, 'Swinging Strike (Blocked)': 1}
pitch['is_swstr'] = pitch['피치결과'].map(swstr_map).fillna(0)

sp_pitch = pitch.merge(first_inning[['gamePk','투수ID']], on=['gamePk','투수ID'])
sp_game = sp_pitch.groupby(['gamePk','투수ID']).agg(
    avg_velo=('구속', 'mean'),
    swstr_rate=('is_swstr', 'mean'),
    pitches=('구속', 'count')
).reset_index()
sp_game = sp_game.merge(results[['gamePk','날짜']], on='gamePk')
sp_game['날짜'] = pd.to_datetime(sp_game['날짜'])
sp_game = sp_game.sort_values(['투수ID','날짜'])

# 최근 3경기 rolling
for pid, grp in sp_game.groupby('투수ID'):
    idx = grp.index
    sp_game.loc[idx, 'velo_r3'] = grp['avg_velo'].shift(1).rolling(3, min_periods=1).mean()
    sp_game.loc[idx, 'swstr_r3'] = grp['swstr_rate'].shift(1).rolling(3, min_periods=1).mean()
    # 구속 변화 (최근3 - 시즌전체)
    season_avg = grp['avg_velo'].expanding().mean().shift(1)
    recent_avg = grp['avg_velo'].shift(1).rolling(3, min_periods=1).mean()
    sp_game.loc[idx, 'velo_trend'] = recent_avg - season_avg

sp_pitch_feat = sp_game[['gamePk','투수ID','velo_r3','swstr_r3','velo_trend']].copy()

# 홈/원정 merge
df = df.merge(sp_pitch_feat.rename(columns={'velo_r3':'home_velo','swstr_r3':'home_swstr','velo_trend':'home_velo_trend'}),
              left_on=['gamePk','홈팀선발ID'], right_on=['gamePk','투수ID'], how='left').drop(columns='투수ID')
df = df.merge(sp_pitch_feat.rename(columns={'velo_r3':'away_velo','swstr_r3':'away_swstr','velo_trend':'away_velo_trend'}),
              left_on=['gamePk','원정팀선발ID'], right_on=['gamePk','투수ID'], how='left').drop(columns='투수ID')
df['swstr_spread'] = df['home_swstr'] - df['away_swstr']
df['velo_spread'] = df['home_velo'] - df['away_velo']

print(f'\n추가 피처 생성 완료: {df.shape}')

In [ ]:
# 셀 10: 추가 피처 단변량 AUC 스크리닝
new_features = [
    'home_sp_homeERA', 'away_sp_awayERA', 'ha_era_spread',
    'home_r5_득점', 'away_r5_실점', 'rdiff_spread',
    'home_streak', 'away_streak', 'streak_spread',
    'home_il', 'away_il', 'il_spread',
    'home_brl', 'away_brl', 'brl_spread',
    'home_velo', 'away_velo', 'velo_spread',
    'home_swstr', 'away_swstr', 'swstr_spread',
    'home_velo_trend', 'away_velo_trend'
]

print("=== 추가 피처 단변량 AUC ===\n")
add_auc = []
for col in new_features:
    valid = df[[col, '홈승']].dropna()
    if len(valid) < 100:
        print(f'  {col}: 데이터 부족 ({len(valid)}건)')
        add_auc.append({'feature':col, 'auc_adj':np.nan, 'n':len(valid)})
        continue
    try:
        auc = roc_auc_score(valid['홈승'], valid[col])
        auc_adj = max(auc, 1-auc)
        direction = '+' if auc >= 0.5 else '-'
        status = '✅ 채택' if auc_adj >= 0.52 else '❌ 탈락'
        print(f'  {col}: AUC={auc_adj:.4f} (n={len(valid)}) {direction} {status}')
        add_auc.append({'feature':col, 'auc_adj':auc_adj, 'n':len(valid), 'status':status})
    except:
        print(f'  {col}: 계산 실패')
        add_auc.append({'feature':col, 'auc_adj':np.nan, 'n':len(valid)})

add_df = pd.DataFrame(add_auc).sort_values('auc_adj', ascending=False)

# 기존 + 추가 통합 정리
print("\n\n=== 전체 피처 최종 AUC 순위 (0.52 이상) ===\n")
all_features = [
    'era_combined', 'wr_spread', 'ops_spread', 'home_sp_K9', 'away_sp_K9',
    'park_rpg', 'home_bp_risk', '기온_F', 'home_sp_WHIP', 'away_sp_WHIP'
] + [f for f in new_features if f in df.columns]

final_auc = []
for col in all_features:
    if col not in df.columns:
        continue
    valid = df[[col, '홈승']].dropna()
    if len(valid) < 100:
        continue
    try:
        auc = roc_auc_score(valid['홈승'], valid[col])
        auc_adj = max(auc, 1-auc)
        miss = df[col].isnull().mean()
        final_auc.append({'feature':col, 'auc_adj':auc_adj, 'n':len(valid), 'missing':f'{miss:.1%}'})
    except:
        continue

final_df = pd.DataFrame(final_auc).sort_values('auc_adj', ascending=False)
print(final_df.to_string(index=False))

In [ ]:
# 셀 11: Phase 4+5 - 결측처리 + 모델링 + 평가
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# === 중복 제거 ===
df_model = df.drop_duplicates(subset='gamePk', keep='first').reset_index(drop=True)
df_model['날짜'] = pd.to_datetime(df_model['날짜'])
df_model['월'] = df_model['날짜'].dt.month
print(f'모델링 테이블: {df_model.shape}')

# === 피처 정의 ===
core_features = ['era_combined', 'rdiff_spread', 'streak_spread',
                 'home_sp_K9', 'home_sp_WHIP', 'wr_spread', 'away_sp_WHIP']
tree_extra = ['home_velo_trend', 'away_velo_trend', 'away_swstr']
all_features = core_features + tree_extra

# === Train/Test 분리 (4~7월 / 8~9월) ===
train = df_model[(df_model['월'] >= 4) & (df_model['월'] <= 7)].copy()
test = df_model[(df_model['월'] >= 8) & (df_model['월'] <= 9)].copy()
print(f'Train: {len(train)} (4~7월)')
print(f'Test: {len(test)} (8~9월)')
print(f'Train 홈승률: {train["홈승"].mean():.4f}')
print(f'Test 홈승률: {test["홈승"].mean():.4f}')

# === 결측 처리 ===
# Core 피처: train median으로 impute
medians = {}
for col in core_features:
    medians[col] = train[col].median()
    train[col] = train[col].fillna(medians[col])
    test[col] = test[col].fillna(medians[col])

# Tree extra: 결측 그대로 (XGBoost/LightGBM이 처리)

print(f'\nTrain 결측 확인 (core): {train[core_features].isnull().sum().sum()}')
print(f'Test 결측 확인 (core): {test[core_features].isnull().sum().sum()}')

# ============================================================
# 모델 1: Logistic Regression (core only)
# ============================================================
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(train[core_features])
X_test_lr = scaler.transform(test[core_features])
y_train = train['홈승'].values
y_test = test['홈승'].values

lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_lr, y_train)
lr_train_prob = lr.predict_proba(X_train_lr)[:, 1]
lr_test_prob = lr.predict_proba(X_test_lr)[:, 1]

print("\n=== Logistic Regression (core 7개) ===")
print(f"Train AUC: {roc_auc_score(y_train, lr_train_prob):.4f}")
print(f"Test AUC:  {roc_auc_score(y_test, lr_test_prob):.4f}")
print(f"Train Brier: {brier_score_loss(y_train, lr_train_prob):.4f}")
print(f"Test Brier:  {brier_score_loss(y_test, lr_test_prob):.4f}")

# 계수 확인
print("\n계수:")
for feat, coef in zip(core_features, lr.coef_[0]):
    print(f"  {feat}: {coef:+.4f}")

# ============================================================
# 모델 2: XGBoost (core + tree_extra)
# ============================================================
try:
    import xgboost as xgb

    X_train_xg = train[all_features].values
    X_test_xg = test[all_features].values

    dtrain = xgb.DMatrix(X_train_xg, label=y_train, feature_names=all_features)
    dtest = xgb.DMatrix(X_test_xg, label=y_test, feature_names=all_features)

    # 보수적 파라미터 (과적합 방지)
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'max_depth': 3,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 20,
        'reg_alpha': 1.0,
        'reg_lambda': 5.0,
        'seed': 42,
        'verbosity': 0
    }

    # Early stopping으로 최적 라운드 찾기
    evals = [(dtrain, 'train'), (dtest, 'test')]
    xg_model = xgb.train(params, dtrain, num_boost_round=500,
                          evals=evals, early_stopping_rounds=30, verbose_eval=False)

    xg_train_prob = xg_model.predict(dtrain)
    xg_test_prob = xg_model.predict(dtest)

    print("\n=== XGBoost (core 7 + extra 3) ===")
    print(f"Best round: {xg_model.best_iteration}")
    print(f"Train AUC: {roc_auc_score(y_train, xg_train_prob):.4f}")
    print(f"Test AUC:  {roc_auc_score(y_test, xg_test_prob):.4f}")
    print(f"Train Brier: {brier_score_loss(y_train, xg_train_prob):.4f}")
    print(f"Test Brier:  {brier_score_loss(y_test, xg_test_prob):.4f}")

    # Feature importance
    imp = xg_model.get_score(importance_type='gain')
    imp_sorted = sorted(imp.items(), key=lambda x: x[1], reverse=True)
    print("\nFeature Importance (gain):")
    for feat, gain in imp_sorted:
        print(f"  {feat}: {gain:.4f}")

except ImportError:
    print("\nXGBoost 미설치. pip install xgboost 후 재실행")
    xg_test_prob = None

# ============================================================
# 모델 3: LightGBM (core + tree_extra)
# ============================================================
try:
    import lightgbm as lgb

    lgb_train = lgb.Dataset(train[all_features], label=y_train)
    lgb_test = lgb.Dataset(test[all_features], label=y_test, reference=lgb_train)

    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'max_depth': 3,
        'learning_rate': 0.05,
        'num_leaves': 8,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_samples': 30,
        'reg_alpha': 1.0,
        'reg_lambda': 5.0,
        'seed': 42,
        'verbose': -1
    }

    callbacks = [lgb.early_stopping(30), lgb.log_evaluation(0)]
    lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=500,
                          valid_sets=[lgb_train, lgb_test], callbacks=callbacks)

    lgb_train_prob = lgb_model.predict(train[all_features])
    lgb_test_prob = lgb_model.predict(test[all_features])

    print("\n=== LightGBM (core 7 + extra 3) ===")
    print(f"Best round: {lgb_model.best_iteration}")
    print(f"Train AUC: {roc_auc_score(y_train, lgb_train_prob):.4f}")
    print(f"Test AUC:  {roc_auc_score(y_test, lgb_test_prob):.4f}")
    print(f"Train Brier: {brier_score_loss(y_train, lgb_train_prob):.4f}")
    print(f"Test Brier:  {brier_score_loss(y_test, lgb_test_prob):.4f}")

    # Feature importance
    imp_lgb = dict(zip(all_features, lgb_model.feature_importance(importance_type='gain')))
    imp_lgb_sorted = sorted(imp_lgb.items(), key=lambda x: x[1], reverse=True)
    print("\nFeature Importance (gain):")
    for feat, gain in imp_lgb_sorted:
        print(f"  {feat}: {gain:.1f}")

except ImportError:
    print("\nLightGBM 미설치. pip install lightgbm 후 재실행")
    lgb_test_prob = None

# ============================================================
# XGBoost core only (extra 없이) 비교
# ============================================================
try:
    X_train_xg_core = train[core_features].values
    X_test_xg_core = test[core_features].values
    dtrain_c = xgb.DMatrix(X_train_xg_core, label=y_train, feature_names=core_features)
    dtest_c = xgb.DMatrix(X_test_xg_core, label=y_test, feature_names=core_features)
    xg_core = xgb.train(params, dtrain_c, num_boost_round=500,
                         evals=[(dtrain_c,'train'),(dtest_c,'test')],
                         early_stopping_rounds=30, verbose_eval=False)
    xg_core_test_prob = xg_core.predict(dtest_c)
    print(f"\n=== XGBoost (core only 7개) ===")
    print(f"Test AUC: {roc_auc_score(y_test, xg_core_test_prob):.4f}")
except:
    pass

In [ ]:
# 셀 12: Phase 6 - 확신도 필터 + A안 비교 + 월별 검증
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

# === 각 모델의 test 확률을 test DataFrame에 붙이기 ===
test = test.copy()
test['lr_prob'] = lr_test_prob
if xg_test_prob is not None:
    test['xg_prob'] = xg_test_prob
if lgb_test_prob is not None:
    test['lgb_prob'] = lgb_test_prob

# === 확신도 필터 함수 ===
def evaluate_threshold(probs, y_true, months, name, target_games=148):
    """A안과 동일 경기수(~148)가 되는 threshold 찾기"""
    # threshold를 올려가며 경기수 맞추기
    best_th = 0.5
    best_diff = 999
    for th in np.arange(0.50, 0.85, 0.005):
        n = ((probs >= th) | (probs <= (1-th))).sum()
        if abs(n - target_games) < best_diff:
            best_diff = abs(n - target_games)
            best_th = th

    mask = (probs >= best_th) | (probs <= (1 - best_th))
    filtered = pd.DataFrame({'prob': probs[mask], 'actual': y_true[mask], 'month': months[mask]})
    filtered['pred'] = (filtered['prob'] >= 0.5).astype(int)
    filtered['correct'] = (filtered['pred'] == filtered['actual']).astype(int)

    overall_acc = filtered['correct'].mean()
    n_games = len(filtered)

    print(f"\n{'='*50}")
    print(f"{name} | Threshold: {best_th:.3f} | Games: {n_games}")
    print(f"Overall Accuracy: {overall_acc:.4f} ({overall_acc*100:.1f}%)")
    print(f"{'='*50}")

    # 월별
    monthly = filtered.groupby('month').agg(
        accuracy=('correct', 'mean'),
        games=('correct', 'count')
    )
    print("\n월별:")
    for m, row in monthly.iterrows():
        print(f"  {m}월: {row['accuracy']:.3f} ({int(row['games'])}경기)")

    if len(monthly) > 1:
        print(f"  월별 편차: {monthly['accuracy'].std():.4f}")

    return overall_acc, n_games, best_th

# === A안 벤치마크: 8~9월 OOS 73.0% (148경기) ===
print("="*60)
print("A안 벤치마크: 8~9월 OOS 73.0% (148경기)")
print("="*60)

y_test_arr = test['홈승'].values
months_arr = test['월'].values

# 각 모델 평가
lr_acc, lr_n, lr_th = evaluate_threshold(
    test['lr_prob'].values, y_test_arr, months_arr, 'Logistic Regression')

if 'xg_prob' in test.columns:
    xg_acc, xg_n, xg_th = evaluate_threshold(
        test['xg_prob'].values, y_test_arr, months_arr, 'XGBoost (full)')

if 'lgb_prob' in test.columns:
    lgb_acc, lgb_n, lgb_th = evaluate_threshold(
        test['lgb_prob'].values, y_test_arr, months_arr, 'LightGBM')

# === 다양한 경기수에서의 적중률 커브 ===
print("\n\n=== 경기수별 적중률 커브 (XGBoost) ===")
if 'xg_prob' in test.columns:
    probs = test['xg_prob'].values
    confidence = np.abs(probs - 0.5)  # 확신도 = |p - 0.5|
    sorted_idx = np.argsort(-confidence)  # 확신도 높은 순

    for top_n in [50, 100, 148, 200, 300, 400, len(test)]:
        if top_n > len(test):
            continue
        idx = sorted_idx[:top_n]
        preds = (probs[idx] >= 0.5).astype(int)
        acc = (preds == y_test_arr[idx]).mean()
        print(f"  Top {top_n:>4}경기: {acc:.4f} ({acc*100:.1f}%)")

In [ ]:
# 셀 13: 시각화 - 캘리브레이션 + 월별 + 학습곡선
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1) 캘리브레이션 커브 (모든 모델)
ax = axes[0, 0]
for name, probs in [('LR', lr_test_prob),
                     ('XGB', xg_test_prob if xg_test_prob is not None else None),
                     ('LGB', lgb_test_prob if lgb_test_prob is not None else None)]:
    if probs is None:
        continue
    fraction_pos, mean_predicted = calibration_curve(y_test, probs, n_bins=10, strategy='uniform')
    ax.plot(mean_predicted, fraction_pos, 's-', label=name)
ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
ax.set_title('Calibration Curve (Test Set)')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.legend()
ax.grid(True, alpha=0.3)

# 2) 확률 분포 (XGBoost)
ax = axes[0, 1]
if xg_test_prob is not None:
    ax.hist(xg_test_prob[y_test == 1], bins=30, alpha=0.6, color='#2ecc71', label='Home Win', density=True)
    ax.hist(xg_test_prob[y_test == 0], bins=30, alpha=0.6, color='#e74c3c', label='Away Win', density=True)
    ax.axvline(0.5, color='black', linestyle='--')
    ax.set_title('XGBoost Probability Distribution (Test)')
    ax.set_xlabel('Predicted P(Home Win)')
    ax.legend()

# 3) 경기수별 적중률 커브 (전 모델 비교)
ax = axes[0, 2]
for name, probs, color in [('LR', lr_test_prob, '#3498db'),
                             ('XGB', xg_test_prob, '#e67e22'),
                             ('LGB', lgb_test_prob, '#9b59b6')]:
    if probs is None:
        continue
    confidence = np.abs(probs - 0.5)
    sorted_idx = np.argsort(-confidence)
    ns = range(30, len(y_test), 10)
    accs = []
    for n in ns:
        idx = sorted_idx[:n]
        preds = (probs[idx] >= 0.5).astype(int)
        accs.append((preds == y_test[idx]).mean())
    ax.plot(ns, accs, '-', label=name, color=color, linewidth=2)
ax.axhline(0.73, color='red', linestyle='--', linewidth=2, label='A안 73%')
ax.axhline(0.543, color='gray', linestyle=':', label='Baseline 54.3%')
ax.set_title('Accuracy by # Games Selected (Test)')
ax.set_xlabel('Number of Games')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.45, 0.85)

# 4) 월별 적중률 비교 (XGBoost, threshold 기준)
ax = axes[1, 0]
if xg_test_prob is not None:
    probs = xg_test_prob
    confidence = np.abs(probs - 0.5)
    # top 148 기준
    sorted_idx = np.argsort(-confidence)[:148]
    test_filtered = test.iloc[sorted_idx]
    test_filtered = test_filtered.copy()
    test_filtered['pred'] = (probs[sorted_idx] >= 0.5).astype(int)
    test_filtered['correct'] = (test_filtered['pred'] == test_filtered['홈승']).astype(int)
    monthly_acc = test_filtered.groupby('월')['correct'].mean()

    ax.bar(monthly_acc.index, monthly_acc.values, color='#3498db', alpha=0.8)
    ax.axhline(0.73, color='red', linestyle='--', label='A안 73%')
    ax.set_title(f'XGBoost Monthly Accuracy (Top 148 games)')
    ax.set_xlabel('Month')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.4, 0.9)
    ax.legend()
    for m, v in monthly_acc.items():
        ax.text(m, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# 5) Train 월별 적중률 (과적합 체크)
ax = axes[1, 1]
train_copy = train.copy()
train_probs_xg = xg_train_prob if xg_test_prob is not None else lr_train_prob
train_copy['pred'] = (train_probs_xg >= 0.5).astype(int)
train_copy['correct'] = (train_copy['pred'] == train_copy['홈승']).astype(int)

confidence_tr = np.abs(train_probs_xg - 0.5)
sorted_idx_tr = np.argsort(-confidence_tr)
# train에서 상위 비율 동일하게
train_top_ratio = 148 / len(test)
train_top_n = int(len(train) * train_top_ratio)
train_filtered = train_copy.iloc[sorted_idx_tr[:train_top_n]]
monthly_train = train_filtered.groupby('월')['correct'].mean()

ax.bar(monthly_train.index, monthly_train.values, color='#e67e22', alpha=0.8)
ax.axhline(0.73, color='red', linestyle='--', label='A안 73%')
ax.set_title(f'XGBoost Monthly Accuracy - TRAIN (Top {train_top_n} games)')
ax.set_xlabel('Month')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.4, 0.9)
ax.legend()
for m, v in monthly_train.items():
    ax.text(m, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# 6) 요약 테이블
ax = axes[1, 2]
ax.axis('off')
summary = f"""
B안 vs A안 최종 비교

A안 (v2 균형형, S1≥4+C≥5)
  4~7월: 73.1% (283경기)
  8~9월: 73.0% (148경기)
  월별 편차: 1.8%p

B안 결과 (이 그래프에서 확인)
  모델: LR / XGBoost / LightGBM
  피처: {len(core_features)} core + {len(tree_extra)} optional
  Train: 4~7월 / Test: 8~9월

판정 기준:
  ✅ Test 73% 이상 → B안 채택
  ✅ Train-Test 차이 5%p 이내
  ✅ 월별 편차 5%p 이내
"""
ax.text(0.05, 0.95, summary, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('/content/phase5_results.png', dpi=150, bbox_inches='tight')
plt.show()